In [ ]:
from itertools import pairwise
from typing import Optional
import warnings

from transformers import pipelines

from trouver.helper.html import HTMLTagWithIndices
from trouver.obsidian.vault import VaultNote
from trouver.machine_learning.tokenize.def_and_notat_token_classification import _make_tag



In [ ]:
from fastcore.test import *

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _current_token_continues_the_previous_token(
        current_token: dict,
        previous_token: dict,
        note: Optional[VaultNote] = None,
        ) -> bool:
    """
    Helper function to `_divide_token_preds_into_parts`.
    """
    if current_token['entity'].startswith('I-'):
        if current_token['entity'][2:] == previous_token['entity'][2:]:
            return True
        elif note:
            warnings.warn(rf"""
                In the note {note.name} at {note.path()},
                The token '{previous_token['word']}' is marked as '{previous_token['entity']}'
                and the subsequent token '{current_token['word']}' is marked as '{current_token['entity']}',
                which is unusual because the two consecutive tokens seem to be of different
                entities, and yet the latter token does not start with a 'B-'.

                The latter token will be treated like the beginning of a new entity."""
                    )
            return False
    else:
        return False
        

In [ ]:
#| hide
previous_token_1 = {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 33,  # This is moot for the purposes of this test.
        'word': '}',
        'start': 61,
        'end': 62}
current_token_1 = {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 34,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 62,
        'end': 63}
assert _current_token_continues_the_previous_token(current_token_1, previous_token_1, note=VaultNote('', rel_path='hi'))

# Something like below should hopefully not happen, but it should still give a warning message
previous_token_2 = {
        'entity': 'I-definition',
        'score': 0.97785944,
        'index': 33,  # This is moot for the purposes of this test.
        'word': '}',
        'start': 61,
        'end': 62}
current_token_2 = {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 34,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 62,
        'end': 63}

with warnings.catch_warnings(record=True) as w:
    sample_output = _current_token_continues_the_previous_token(current_token_2, previous_token_2, note=VaultNote('', rel_path='hi'))
    assert w
    assert not sample_output

previous_token_3 = {
        'entity': 'I-definition',
        'score': 0.97785944,
        'index': 33,  # This is moot for the purposes of this test.
        'word': '##tion',
        'start': 58,
        'end': 62}
current_token_3 = {
        'entity': 'B-notation',
        'score': 0.97785944,
        'index': 34,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 62,
        'end': 63}

assert not _current_token_continues_the_previous_token(current_token_3, previous_token_3, note=VaultNote('', rel_path='hi'))


In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _divide_token_preds_into_parts(
        token_preds: list[dict[str]],
        excessive_space_threshold: int,
        note: Optional[VaultNote] = None
        ) -> list[list[dict[str]]]:
    """
    Divide `token_preds` into parts so that each part
    represents a single definition/notation marking.

    Helper function to `_html_tags_from_token_preds`.
    """
    token_preds_parts = []
    for current_token in token_preds:
        if not token_preds_parts:
            token_preds_parts.append([current_token])
            continue
        prev_token = token_preds_parts[-1][-1]
        if _current_token_continues_the_previous_token(
                current_token, prev_token, note):
            prev_token_end = prev_token['end']
            cur_token_start = current_token['start']
            if prev_token_end + excessive_space_threshold >= cur_token_start and note:
                Warning(rf"""
                    In the note {note.name} at {note.path()},
                    There seems to be excessive space between the token
                    {prev_token['word']} and {current_token['word']}, which
                    seem to be part of the same entity"""
                        )
            token_preds_parts[-1].append(current_token)
        else:
            token_preds_parts.append([current_token])
    return token_preds_parts

In [ ]:
#| hide

main_text = r"Let $I \subset A$ be an ideal. Define its radical by $\sqrt{I}$"

preds = [
    {
        'entity': 'B-definition',
        'score': 0.37319255,
        'index': 25,  # This is moot for the purposes of this test.
        'word': 'radical',
        'start': 42,
        'end': 49
    },
    {
        'entity': 'B-notation',
        'score': 0.67021805,
        'index': 27,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 53,
        'end': 54},
    {
        'entity': 'I-notation',
        'score': 0.9748327,
        'index': 28,  # This is moot for the purposes of this test.
        'word': '\\',
        'start': 54,
        'end': 55},
    {
        'entity': 'I-notation',
        'score': 0.9754836,
        'index': 29,  # This is moot for the purposes of this test.
        'word': 'sq',
        'start': 55,
        'end': 57},
    {
        'entity': 'I-notation',
        'score': 0.9750675,
        'index': 30,  # This is moot for the purposes of this test.
        'word': '##rt',
        'start': 57,
        'end': 59},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 31,  # This is moot for the purposes of this test.
        'word': '{',
        'start': 59,
        'end': 60},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 32,  # This is moot for the purposes of this test.
        'word': 'i',
        'start': 60,
        'end': 61},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 33,  # This is moot for the purposes of this test.
        'word': '}',
        'start': 61,
        'end': 62},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 34,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 62,
        'end': 63},
    ]

output = _divide_token_preds_into_parts(
    preds,  excessive_space_threshold=2, note=VaultNote('', rel_path='hi')
)

# Test that the list finds two parts, one for the definition, and the other for the notation.
test_eq(len(output), 2)
test_eq(len(output[0]), 1)
test_eq(len(output[1]), len(preds) - 1)

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _html_tag_data_from_part(
        main_text: str,
        # part: list[dict[str]]) -> tuple[bs4.element.Tag, int, int]:
        part: list[dict[str]] # An item of an output of `_divide_token_preds_into_parts`. Each dict likely contains keys such as `'entity'`, `'score'`, `'index'`, `'word'`, `'start'`, and `'end'`, depending on the model used.
        ) -> tuple[HTMLTagWithIndices]:
    """
    Helper function to `_html_tags_from_token_preds`
    """
    start_token: dict[str] = part[0]
    end_token: dict[str] = part[-1]
    start_char_pos: int = start_token['start']
    end_char_pos: int = end_token['end']

    # Depending on the tokenizer starting spaces might be included in a given token.
    # We exclude such a starting space.
    while main_text[start_char_pos].isspace():
        start_char_pos += 1
    while main_text[end_char_pos-1].isspace():
        end_char_pos -= 1

    # the `'entity'` is either 'I-definition', 'B-definition', 'I-notation',
    # or 'B-notation'
    entity_type = start_token['entity'][2:]
    html_text = main_text[start_char_pos:end_char_pos]
    
    # return (_make_tag(html_text, entity_type), start_char, end_char)
    return HTMLTagWithIndices(_make_tag(html_text, entity_type), start_char_pos, end_char_pos)

In [ ]:
#| hide

main_text = r"Let $I \subset A$ be an ideal. Define its radical by $\sqrt{I}$"

sample_output_1 = _html_tag_data_from_part(
    main_text, [{
        'entity': 'B-definition',
        'score': 0.37319255,
        'index': 25,  # This is moot for the purposes of this test.
        'word': 'radical',
        'start': 42,
        'end': 49
    }])
test_eq(str(sample_output_1.tag), '<b definition="" style="border-width:1px;border-style:solid;padding:3px">radical</b>')

sample_output_2 = _html_tag_data_from_part(
    main_text, [{
        'entity': 'B-notation',
        'score': 0.67021805,
        'index': 27,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 53,
        'end': 54},
    {
        'entity': 'I-notation',
        'score': 0.9748327,
        'index': 28,  # This is moot for the purposes of this test.
        'word': '\\',
        'start': 54,
        'end': 55},
    {
        'entity': 'I-notation',
        'score': 0.9754836,
        'index': 29,  # This is moot for the purposes of this test.
        'word': 'sq',
        'start': 55,
        'end': 57},
    {
        'entity': 'I-notation',
        'score': 0.9750675,
        'index': 30,  # This is moot for the purposes of this test.
        'word': '##rt',
        'start': 57,
        'end': 59},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 31,  # This is moot for the purposes of this test.
        'word': '{',
        'start': 59,
        'end': 60},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 32,  # This is moot for the purposes of this test.
        'word': 'i',
        'start': 60,
        'end': 61},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 33,  # This is moot for the purposes of this test.
        'word': '}',
        'start': 61,
        'end': 62},
    {
        'entity': 'I-notation',
        'score': 0.97785944,
        'index': 34,  # This is moot for the purposes of this test.
        'word': '$',
        'start': 62,
        'end': 63},
    ])
test_eq(str(sample_output_2.tag), r'<span notation="" style="border-width:1px;border-style:solid;padding:3px">$\sqrt{I}$</span>')
# main_text.find('radical')

In [ ]:
#| hide

# With modernbert, the tokenizer tends to capture spaces, within tokens, which creates some subtle issues with spacing.
text = r'Definition 1.1. - We call $\mathscr{U}$-topos, or simply topos if there is no fear of confusion, a category $\mathrm{E}$ such that there is a site $\mathrm{C} \in \mathscr{U}$ such that $\mathrm{E}$ is equivalent to the category $\tilde{\mathrm{C}}$ of $\mathrm{U}$-sheaves of sets over $\mathrm{C}$ .'

part_1 = [
        {'entity': 'B-definition',
        'score': 0.9991014,
        'index': 9,
        'word': 'Ġ$\\',
        'start': 25,
        'end': 28},
        {'entity': 'I-definition',
        'score': 0.99974686,
        'index': 10,
        'word': 'mathscr',
        'start': 28,
        'end': 35},
        {'entity': 'I-definition',
        'score': 0.999652,
        'index': 11,
        'word': '{',
        'start': 35,
        'end': 36},
        {'entity': 'I-definition',
        'score': 0.99985695,
        'index': 12,
        'word': 'U',
        'start': 36,
        'end': 37},
        {'entity': 'I-definition',
        'score': 0.9997553,
        'index': 13,
        'word': '}$-',
        'start': 37,
        'end': 40},
        {'entity': 'I-definition',
        'score': 0.9997918,
        'index': 14,
        'word': 'top',
        'start': 40,
        'end': 43},
        {'entity': 'I-definition',
        'score': 0.9997335,
        'index': 15,
        'word': 'os',
        'start': 43,
        'end': 45},]

part_2 = [
        {'entity': 'B-definition',
        'score': 0.594063,
        'index': 19,
        'word': 'Ġto',
        'start': 56,
        'end': 59},
        {'entity': 'I-definition',
        'score': 0.9056522,
        'index': 20,
        'word': 'pos',
        'start': 59,
        'end': 62}]

sample_output_1: HTMLTagWithIndices = _html_tag_data_from_part(text, part_1)
assert not sample_output_1.tag.text.startswith(' ')
assert sample_output_1.tag.text == '$\\mathscr{U}$-topos'

sample_output_2: HTMLTagWithIndices = _html_tag_data_from_part(text, part_1)
assert not sample_output_2.tag.text.startswith(' ')
assert sample_output_2.tag.text == '$\\mathscr{U}$-topos'

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _html_tags_from_token_preds(
        main_text: str,
        token_preds: list[dict[str]], # An output of `pipeline(text)`; Each dict likely contains keys such as `'entity'`, `'score'`, `'index'`, `'word'`, `'start'`, and `'end'`, depending on the model used.
        excessive_space_threshold: int,
        note: Optional[VaultNote] = None,
        ) -> list[HTMLTagWithIndices]:  # Tag element, start, end, where main_text[start:end] needs to be replaced by the tag element.
        # ) -> list[tuple[bs4.element.Tag, int, int]]:  # Tag element, start, end, where main_text[start:end] needs to be replaced by the tag element.
    """
    Return HTML tags for definition and notation classification.

    Helper function to `auto_mark_def_and_notats`.
    """
    parts: list[list[dict[str]]] = _divide_token_preds_into_parts(
        token_preds, excessive_space_threshold, note)
    return [_html_tag_data_from_part(main_text, part) for part in parts]

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _divide_main_text(
        main_text: str,
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline that is used to predict whether tokens are part of definitions or notations introduced in the text. Here, the tokenizer of this pipeline is used to estimate how many tokens a piece of subtext will have.
        # ) -> list[tuple[str, int, int]]:  # The str is a chunk of text, the first int is the index in `main_text` that the chunk starts at, and the second int is the approximate token length of the text. Appending all the chunks of text as they are should result back in the original text.
        ) -> list[tuple[int, int]]:  # Each tuple is a start and end range for pieces of `main_text` to be considered for predictions
    """Divides `main_text` so that predictions can be made on smaller chunks of text.
    
    Assumes that dividing `main_text` along newline characters `\n` will result in
    pieces that are "not too long".

    Helper function to `_format_main_text_and_add_html_tag_data`.


    """
    main_text.split('\n')
    tokenizer = pipeline.tokenizer
    newline_indices = [i for i, char in enumerate(main_text) if char == '\n']
    newline_indices.insert(0, 0)
    chunks = []  # list[tuple[str, int, int]]  # The str is a chunk of text, the first int is the index in `main_text` that the chunk starts at, and the second int is the approximate token length of the text. Appending all the chunks of text as they are should result back in the original text.
    for start, end in pairwise(newline_indices):
        chunk = main_text[start:end]
        chunks.append((chunk, start, len(tokenizer(chunk)['input_ids'])))
    last_chunk = main_text[newline_indices[-1]:]
    chunks.append((last_chunk, newline_indices[-1], len(tokenizer(last_chunk)['input_ids'])))
    return _find_places_to_divide_from_chunks(chunks, pipeline)

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _find_places_to_divide_from_chunks(
        chunks: list[tuple[str, int, int]], # The str is a chunk of text, the first int is the index in `main_text` that the chunk starts at, and the second int is the approximate token length of the text. Appending all the chunks of text as they are should result back in the original text.
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline that is used to predict whether tokens are part of definitions or notations introduced in the text. Here, the tokenizer of this pipeline is used to estimate how many tokens a piece of subtext will have.
        ) -> list[tuple[int, int]]: # Each tuple is a start and end range for pieces of `main_text` to be considered for predictions
    """Identify appropriate indices in `main_text` where (overlapping)
    pieces in `main_text` should start/end for predictions with `pipeline`.
    
    Helper function to `_divide_main_text`.

    We describe how this function is implemented: starting at the first chunk
    (chunks are non-overlapping), start to consider consecutive chunks to
    make up a piece. So maybe we have chunks

        A B C D E F ....

    We build a piece chunk-by-chunk, considering the total token length of the
    built sub-piece along the way. The first chunk within a sub-piece 
    that makes the sub-piece of token-length greater than half the max
    token length with respect to `pipeline.tokenizer` will become the start of the
    next piece, unless the very first chunk in the piece is already longer than half the max
    token length with respect to the tokenizer (this is to ensure that the
    piece-building process does not keep starting at the same chunk).
    Moreover, a piece will stop building as soon as its token-length exceeds
    the max length of the tokenizer.

    For instance, maybe the max length (`tokenizer.model_max_length`)
    for the tokenizer is 512, and the chunks
    are of the following length:

        A   B    C  D   E    F     ...
        76  130  70 13  150  140   ...

    We first build the piece starting at A:

        A
        76

    We continue building the piece by "appending" B:

        A   B
        76  130

    Once we append C as well, the piece's length is now 276 and hence over half of 512,
    so the next piece will start at C: 

        A   B    C
        76  130  70

    Subsequently, we continue building the piece. Only once F is appended does the 
    length of the entire piece exceed 512 (the length is 579):

        A   B    C  D   E    F
        76  130  70 13  150  140
    
    And then we begin building the next piece from C.

    Also, consider an example where the first chunk's length exceeds half the max length
    of the tokenizer:

        A   B    C   ...
        300 200  100 ...

    Here, the first piece will consist of the chunks A, B, and C because
    the length of the piece exceeds the max length of 512 only after appending C.
    To guarantee that the next piece does not start with the chunk A again, B is 
    used as the first chunk in the next piece:

        B    C   ...
        200  100 ...

    If any chunk's token length exceeds the tokenizer's max_model_length, then
    the pipeline/model can only predict on the starting tokens in the chunk. 
    As such, the chunks must not be "too long" for best results on the model's predictions.
    """
    tokenizer = pipeline.tokenizer
    start_chunk_index, next_piece_start_chunk_index = 0, 0
    current_piece_token_len = 0
    pieces_start_and_end = []
    i = 0
    while i < len(chunks):
        chunk = chunks[i]
        current_piece_token_len = current_piece_token_len + chunk[2]
        if (current_piece_token_len > tokenizer.model_max_length / 2
                and start_chunk_index == next_piece_start_chunk_index):
            # Mark where the next piece should start
            next_piece_start_chunk_index = i if start_chunk_index != i else i+1
        if (current_piece_token_len > tokenizer.model_max_length):
            _append_to_pieces_start_and_end(
                pieces_start_and_end, chunks[start_chunk_index], chunk)
            i, start_chunk_index = (
                next_piece_start_chunk_index, next_piece_start_chunk_index)
            current_piece_token_len = 0
            continue
        i += 1
    # Add the last chunk at the end
    _append_to_pieces_start_and_end(
        pieces_start_and_end, chunks[start_chunk_index], chunks[-1])
    return pieces_start_and_end


In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def def_and_notat_preds_by_model(
        text: str,  
        pipeline # The pipeline object created using the token classification model and its tokenizer
        ) -> list[HTMLTagWithIndices]: # HTMLTAgWithIndices consists of an HTML tag carrying the data of the prediction and ints marking where in `text` the definition or notation is at.
    """
    Predict where definitions and notations occur in `text`

    This function uses some of the same helper functions as
    `auto_mark_def_and_notats`, but does not raise warning messages as
    in `auto_mark_def_and_notats`.
    """
    tag_data = _html_tags_from_token_preds(text, pipeline(text), 2, None)
    return tag_data